In [1]:
!pip install -q langchain langchain-community langchain-huggingface \
    sentence-transformers faiss-cpu pypdf chromadb groq gradio langgraph


In [2]:
# SPML Task 1 - Applied ML
# RAG pipeline for querying research papers
# basically: load pdfs -> chunk -> embed -> store in faiss -> retrieve + generate

# run this first if you're on colab
# !pip install -q langchain langchain-community langchain-huggingface
# !pip install -q sentence-transformers faiss-cpu pypdf chromadb
# !pip install -q groq gradio langgraph

import os
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional, TypedDict

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from groq import Groq
from langgraph.graph import StateGraph, END

import gradio as gr


# ------- config -------
GROQ_API_KEY = "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"

EMBEDDING_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
GROQ_MODEL = "llama-3.3-70b-versatile"
# chunk size = how big each text piece is, overlap = how much they share at edges
# overlap helps avoid cutting a sentence right at a boundary
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200

TOP_K            = 5          # how many chunks to pull per query
FAISS_INDEX_PATH = "faiss_index"

# all 7 papers from the task sheet
PAPER_URLS = {
    "Attention Is All You Need": "https://arxiv.org/pdf/1706.03762.pdf",
    "BERT"                     : "https://arxiv.org/pdf/1810.04805.pdf",
    "GPT-3"                    : "https://arxiv.org/pdf/2005.14165.pdf",
    "RAG"                      : "https://arxiv.org/pdf/2005.11401.pdf",
    "Sentence-BERT"            : "https://arxiv.org/pdf/1908.10084.pdf",
    "LoRA"                     : "https://arxiv.org/pdf/2106.09685.pdf",
    "LLaMA 2"                  : "https://arxiv.org/pdf/2307.09288.pdf",
}


# ------- download papers -------

def download_papers(output_dir="papers"):

    import urllib.request
    os.makedirs(output_dir, exist_ok=True)
    paths = []

    for name, url in PAPER_URLS.items():
        fname    = re.sub(r"[^a-z0-9]+", "_", name.lower()) + ".pdf"
        filepath = os.path.join(output_dir, fname)

        if not os.path.exists(filepath):
            print(f"downloading: {name}...")
            urllib.request.urlretrieve(url, filepath)
        else:
            print(f"already have: {name}")

        paths.append(filepath)

    return paths


# ------- load + chunk pdfs -------

def load_and_split_pdfs(pdf_paths: List[str]) -> List[Document]:
    """
    Reads each pdf page by page and splits into smaller chunks.

    We can't just dump a whole 50-page paper into the LLM - context window isn't
    big enough. Instead we store chunks and only retrieve the relevant ones.

    RecursiveCharacterTextSplitter tries paragraph breaks first, then newlines,
    then sentences - so it doesn't cut in weird places as much.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size    = CHUNK_SIZE,
        chunk_overlap = CHUNK_OVERLAP,
        separators    = ["\n\n", "\n", ". ", " ", ""],
    )

    all_docs = []

    for pdf_path in pdf_paths:
        print(f"loading: {pdf_path}")
        loader = PyPDFLoader(pdf_path)
        pages  = loader.load()

        # store the paper name in metadata so we can cite it later
        paper_name = Path(pdf_path).stem.replace("_", " ").title()
        for page in pages:
            page.metadata["source"] = paper_name

        chunks = splitter.split_documents(pages)
        all_docs.extend(chunks)
        print(f"  {len(pages)} pages -> {len(chunks)} chunks")

    print(f"\ntotal chunks: {len(all_docs)}")
    return all_docs


# ------- vector store -------

def build_or_load_vectorstore(docs: Optional[List[Document]] = None) -> FAISS:
    """
    Builds a FAISS index from the chunks, or loads one if it already exists.

    The embedding model converts text -> 384d vectors. Similar meanings end up
    close together in that space, which is what lets us do semantic search

    FAISS is basically an efficient nearest-neighbor search lib - finds the
    closest vectors to your query vector really fast.

    Saving the index means we don't have to re-embed everything on every run,
    which saves a lot of time.
    """
    print("loading embedding model...")
    embeddings = HuggingFaceEmbeddings(
        model_name    = EMBEDDING_MODEL,
        model_kwargs  = {"device": "cpu"},    # swap to "cuda" if you have a gpu
        encode_kwargs = {"normalize_embeddings": True},
    )

    if os.path.exists(FAISS_INDEX_PATH) and docs is None:
        print(f"loading saved index from '{FAISS_INDEX_PATH}'...")
        vectorstore = FAISS.load_local(
            FAISS_INDEX_PATH,
            embeddings,
            allow_dangerous_deserialization=True,
        )
    else:
        print("building faiss index... (might take a minute)")
        vectorstore = FAISS.from_documents(docs, embeddings)
        vectorstore.save_local(FAISS_INDEX_PATH)
        print(f"index saved to '{FAISS_INDEX_PATH}'")

    return vectorstore


# ------- groq llm wrapper -------

class GroqLLM:

    def __init__(self, api_key: str, model: str = GROQ_MODEL):
        self.client = Groq(api_key=api_key)
        self.model  = model

    def generate(self, prompt: str, max_tokens: int = 1024) -> str:
        resp = self.client.chat.completions.create(
            model       = self.model,
            messages    = [{"role": "user", "content": prompt}],
            max_tokens  = max_tokens,
            temperature = 0.2,   # low = more factual, less random
        )
        return resp.choices[0].message.content.strip()


# ------- prompts -------

# main rag prompt - tells the llm to only use what's in the context
# and to cite where it got the info from
RAG_PROMPT_TEMPLATE = """You are a research assistant helping with NLP and ML papers.
Use the context below to answer the question. Only use what's in the context - if it's
not there, say so. Cite the paper names where relevant.

CONTEXT:
{context}

QUESTION:
{question}

For comparison questions, use clear headings to separate the two sides.

ANSWER:"""

# used to figure out what kind of question it is before retrieval
QUERY_TYPE_PROMPT = """Classify this question into one of:
- CONCEPTUAL (asking how something works)
- PAPER_SPECIFIC (about a particular paper)
- COMPARISON (comparing two papers/methods)

Question: {question}

Reply with just the category name."""


# ------- langgraph state -------
# this dict gets passed between every node in the graph
# each node reads from it and adds its output back into it

class RAGState(TypedDict):
    question      : str
    query_type    : Optional[str]            # CONCEPTUAL / PAPER_SPECIFIC / COMPARISON
    retrieved_docs: Optional[List[Document]] # raw chunks from faiss
    context       : Optional[str]            # formatted text from those chunks
    answer        : Optional[str]
    sources       : Optional[List[str]]      # paper names used


# ------- graph nodes -------
# each node is a function that takes state, does one thing, returns updated state
# langgraph calls them in the order we wire up the edges

def make_classify_node(llm: GroqLLM):
    """
    First node - figures out what type of question it is.
    Comparison queries need more chunks retrieved (from different papers),
    so we need to know this before hitting the vector store.
    """
    def classify_query(state: RAGState) -> RAGState:
        prompt = QUERY_TYPE_PROMPT.format(question=state["question"])
        label  = llm.generate(prompt, max_tokens=10).strip().upper()

        # sometimes the model returns something unexpected, default to conceptual
        if label not in ("CONCEPTUAL", "PAPER_SPECIFIC", "COMPARISON"):
            label = "CONCEPTUAL"

        print(f"query type: {label}")
        return {**state, "query_type": label}

    return classify_query


def make_retrieve_node(vectorstore: FAISS):
    """
    Second node - semantic search against the faiss index.

    Using MMR (maximal marginal relevance) instead of plain similarity search.
    MMR tries to get you chunks that are relevant but also diverse - so you don't
    just get 5 very similar chunks saying the same thing.

    For comparison questions we double the k so we actually get chunks from
    both papers being compared.
    """
    def retrieve_docs(state: RAGState) -> RAGState:
        k = TOP_K * 2 if state.get("query_type") == "COMPARISON" else TOP_K

        retriever = vectorstore.as_retriever(
            search_type   = "mmr",
            search_kwargs = {"k": k, "fetch_k": k * 3},
        )

        docs = retriever.invoke(state["question"])
        print(f"retrieved {len(docs)} chunks")
        return {**state, "retrieved_docs": docs}

    return retrieve_docs


def make_format_context_node():
    """
    Third node - just formats the raw chunks into a readable string.
    Labels each chunk with source paper and page number so the llm
    (and us) can trace where things came from.
    """
    def format_context(state: RAGState) -> RAGState:
        docs = state["retrieved_docs"]

        if not docs:
            return {**state, "context": "No relevant context found.", "sources": []}

        parts   = []
        sources = []

        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get("source", "Unknown")
            page   = doc.metadata.get("page", "?")

            if source not in sources:
                sources.append(source)

            parts.append(f"[Chunk {i} | {source} | p.{page}]\n{doc.page_content}")

        context = "\n\n" + "\n\n".join(parts)
        return {**state, "context": context, "sources": sources}

    return format_context


def make_generate_node(llm: GroqLLM):
    """
    Fourth (last) node - sends the retrieved context + question to the LLM.

    The whole point of RAG is here - instead of asking the LLM from memory
    (where it might hallucinate), we're giving it the actual text from the papers
    and asking it to answer from that. Much more reliable for factual questions.
    """
    def generate_answer(state: RAGState) -> RAGState:
        prompt = RAG_PROMPT_TEMPLATE.format(
            context  = state["context"],
            question = state["question"],
        )
        answer = llm.generate(prompt, max_tokens=1024)
        print(f"answer generated ({len(answer)} chars)")
        return {**state, "answer": answer}

    return generate_answer


# ------- build the graph -------

def build_rag_graph(vectorstore: FAISS, llm: GroqLLM):
    """
    Wires everything together as a langgraph pipeline:
        classify -> retrieve -> format_context -> generate -> END

    Langgraph handles passing state between nodes automatically.
    If you want to add conditional branching later (e.g. different retrieval
    for comparison vs conceptual), you'd add conditional edges here.
    """
    graph = StateGraph(RAGState)

    graph.add_node("classify",       make_classify_node(llm))
    graph.add_node("retrieve",       make_retrieve_node(vectorstore))
    graph.add_node("format_context", make_format_context_node())
    graph.add_node("generate",       make_generate_node(llm))

    graph.set_entry_point("classify")
    graph.add_edge("classify",       "retrieve")
    graph.add_edge("retrieve",       "format_context")
    graph.add_edge("format_context", "generate")
    graph.add_edge("generate",        END)

    return graph.compile()


# ------- query function -------

def query_rag(question: str, rag_app) -> Tuple[str, List[str]]:
    """
    Main function to call - pass a question, get back (answer, sources).
    """
    initial_state: RAGState = {
        "question"      : question,
        "query_type"    : None,
        "retrieved_docs": None,
        "context"       : None,
        "answer"        : None,
        "sources"       : None,
    }

    final_state = rag_app.invoke(initial_state)
    answer  = final_state.get("answer",  "No answer generated.")
    sources = final_state.get("sources", [])
    return answer, sources


# ------- gradio ui -------

def launch_ui(rag_app):
    """
    Simple chat interface with gradio.
    gradio auto-generates a public URL when running in colab (share=True),
    which makes it easy to show/demo without setting up anything extra.
    """
    def chat_fn(user_message: str, history: list):
        if not user_message.strip():
            return history, ""

        answer, sources = query_rag(user_message, rag_app)

        # append source info at the bottom
        if sources:
            answer += "\n\n📚 **Sources:** " + " | ".join(sources)

        history.append((user_message, answer))
        return history, ""

    examples = [
        "How does self-attention differ from recurrence?",
        "What problem does RAG solve?",
        "How does LoRA reduce training cost?",
        "Difference between BERT and GPT",
        "What is the key contribution of Sentence-BERT?",
        "How does LLaMA 2 compare to GPT-3?",
    ]

    with gr.Blocks(title="SPML RAG - Paper Q&A") as demo:
        gr.Markdown("## SPML RAG Pipeline\nAsk anything about the 7 research papers.")

        chatbot = gr.Chatbot(height=500)
        with gr.Row():
            msg = gr.Textbox(placeholder="Ask about the papers...", scale=4)
            btn = gr.Button("Ask", variant="primary", scale=1)

        gr.Examples(examples=examples, inputs=msg)

        btn.click(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, msg])
        msg.submit(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, msg])

    demo.launch(share=True)


# ------- main -------

def main():
    print("setting up rag pipeline...")

    # 1. get the pdfs
    pdf_paths = download_papers("papers")

    # 2. load and chunk them
    docs = load_and_split_pdfs(pdf_paths)

    # 3. build the vector index (or load cached one)
    vectorstore = build_or_load_vectorstore(docs)

    # 4. init the llm
    llm = GroqLLM(api_key=GROQ_API_KEY)

    # 5. build pipeline
    rag_app = build_rag_graph(vectorstore, llm)

    print("\npipeline ready!")

    # quick test to make sure things work end to end
    q = "What is the core idea behind the Transformer architecture?"
    print(f"\ntest query: {q}")
    ans, srcs = query_rag(q, rag_app)
    print(f"answer: {ans[:300]}...")
    print(f"sources: {srcs}\n")

    # launch the ui
    launch_ui(rag_app)

    # returning rag_app so you can call query_rag() directly in colab cells too
    return rag_app


if __name__ == "__main__":
    rag_app = main()


# ------- usage in colab -------
# after running main() you can also do direct queries in a cell:
#
#   ans, srcs = query_rag("How does LoRA reduce training cost?", rag_app)
#   print(ans)
#   print("from:", srcs)

/tmp/ipykernel_3396/1233825448.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


setting up rag pipeline...
already have: Attention Is All You Need
already have: BERT
already have: GPT-3
already have: RAG
already have: Sentence-BERT
already have: LoRA
already have: LLaMA 2
loading: papers/attention_is_all_you_need.pdf
  15 pages -> 52 chunks
loading: papers/bert.pdf
  16 pages -> 83 chunks
loading: papers/gpt_3.pdf
  75 pages -> 317 chunks
loading: papers/rag.pdf
  19 pages -> 92 chunks
loading: papers/sentence_bert.pdf
  11 pages -> 60 chunks
loading: papers/lora.pdf
  26 pages -> 111 chunks
loading: papers/llama_2.pdf
  77 pages -> 343 chunks

total chunks: 1058
loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


building faiss index... (might take a minute)
index saved to 'faiss_index'

pipeline ready!

test query: What is the core idea behind the Transformer architecture?
query type: CONCEPTUAL
retrieved 5 chunks
answer generated (1015 chars)
answer: The core idea behind the Transformer architecture is not explicitly stated in the provided context. However, based on the information given in Chunk 3 from the paper "Attention Is All You Need", we can infer that the Transformer architecture makes heavy use of self-attention and multi-head attention...
sources: ['Lora', 'Llama 2', 'Attention Is All You Need', 'Gpt 3']



/tmp/ipykernel_3396/1233825448.py:392: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500)
/tmp/ipykernel_3396/1233825448.py:392: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a37844637ef04c5bf4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
